In [ ]:
import os
from dotenv import load_dotenv


In [ ]:
load_dotenv()

In [ ]:
if os.environ['OPENAI_API_KEY']:
    print("OPENAI key is available")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

In [ ]:
llm = ChatOpenAI(model = "gpt-5-nano", temperature=0)

In [ ]:
response = llm.invoke("What is capital of India")

In [ ]:
response.content

# RAG implementation with own data

### Step 1: Preparing documents

In [ ]:
from langchain_core.documents import Document

In [ ]:
my_text = """Delhi, India’s capital territory, is a massive metropolitan area in the country’s north. In Old Delhi,
a neighborhood dating to the 1600s, stands the imposing Mughal-era Red Fort, a symbol of India, and the sprawling Jama Masjid mosque, 
whose courtyard accommodates 25,000 people. Nearby is Chandni Chowk, a vibrant bazaar filled with food carts, sweets shops and spice stalls"""

docs = [Document(page_content = my_text, metadata ={"source":"text","DocumentID": "Doc1"})]
docs

### Step 2: chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.split_documents(docs)
chunks

### Step 3: Creating embeddings and storing in vector database

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small")

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

### Step 4: Semantic search

In [ ]:
vectorstore.similarity_search("Where is Delhi located in India?", k=2)

In [ ]:
context = vectorstore.similarity_search("Where is Delhi located in India?", k=2)

In [ ]:
llm.invoke(f"Where is Delhi located in India? use only this context and no llm search outside this: {context}")